# Notebook 06 — Full Model and Leave-One-Out Ablation Matrix

**Summary:** Closes the loop — assembles the full proposed model and runs the complete
leave-one-out ablation sweep. Produces the master results table for the thesis.

**Full model components:**
- C1: R-Blur foveal transform (`src/foveation.py`)
- C2: Trace-based metameric periphery (Watson-SNR grounded)
- C3: VOneBlock Poisson noise (`src/overrides.py`)
- C4: ResNet-50 ventral back-end (`src/models.py`)
- C5: Multi-glance + confidence halting (`src/it_feedback.py`)

**Outputs:** `results/06_ablation_table.json` · `results/06_ablation_table.png` ·
`results/06_efficiency_curve.png`

---

## Mathematics

**Marginal contribution** of component $C_k$:

$$
\Delta_k = \mathrm{metric}(\text{full model}) - \mathrm{metric}(\text{full} \setminus C_k)
$$

**EOT adversarial gradient** (for stochastic C3):

$$
\bar g = \frac{1}{N}\sum_{i=1}^N \nabla_x \ell(f(T_i(x)),\, y), \quad N\ge10
$$

**Efficiency curve:** accuracy $A$ vs average glances $G$:

$$
\eta = \frac{A}{G} \quad \text{(accuracy-per-glance efficiency)}
$$

> **Smoke-test scope:** All models use random weights; accuracy numbers are not
> meaningful. The notebook demonstrates the full pipeline and ablation machinery.


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else: PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import common, datasets, models
from src.foveation import (SNRProfile, build_foveated_transform,
                            apply_rblur, FoveatedTransform, TraceBasedPeriphery)
from src.overrides  import apply_vone_block_input_gradient_fix, set_v1_noise_mode
from src.it_feedback import FixationGrid, MultiGlanceFoveatedModel
from src.eval_harness import pgd_attack

CFG = {
    "seed": 42, "smoke_test": True,
    "image_size": 32, "n_classes": 10, "ppd": 4.0,
    "fovea_deg": 1.0, "transition_deg": 0.5,
    "rblur_sigma0": 0.5, "rblur_slope": 1.5,
    "snr0_db": 30.0, "beta": 2.0, "patch_size": 8,
    "max_glances": 4, "halt_threshold": 0.75, "margin": 0.25,
    # Beta sweep for the "full" model (VOneNet + trace periphery + multi-glance
    # combined) -- same range as 02_foveation_rblur_and_periphery.ipynb's visual
    # sweep, since beta=1-3 alone barely change alpha(r) within CIFAR-10's 8deg
    # field of view (see chat discussion / that notebook's beta-sweep figure).
    "beta_sweep": [1.0, 2.0, 3.0, 5.0, 8.0],
    "n_batches_c10c_smoke": 3,   # capped CIFAR-10-C batches/corruption/severity in smoke_test
    # Training
    "n_epochs_smoke": 3, "n_batches_smoke": 20,
    "n_epochs_full": 20, "n_batches_full": None,
    "batch_size": 32, "lr": 1e-3,
    "data_dir": os.path.join(PROJECT_ROOT, "data"),
    "cifar10c_dir": os.path.join(PROJECT_ROOT, "data", "cifar10c"),
    # Adversarial
    "eps": 4/255, "pgd_steps": 5, "pgd_alpha": 1/255, "eot_samples": 4,
    "result_dir": os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":   os.path.join(PROJECT_ROOT, "checkpoints"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
apply_vone_block_input_gradient_fix()
print(f"Device: {device} | Smoke test: {CFG['smoke_test']}")


## Assembling the Full Model

The full model pipeline:

```
x_raw  (raw [0,1] pixels)
  ↓  FoveatedTransform (C1: R-Blur  +  C2: trace-based periphery)
  ↓  NormalizedModel(VOneResNet-50, C3: Poisson noise, C4: ResNet-50 back-end)
  ↓  MultiGlanceFoveatedModel (C5: multi-glance + confidence halting)
  →  logits
```

For leave-one-out: each ablation removes exactly one component and keeps all others.


In [ ]:
# ── Build the full model and all ablation variants ───────────────────────

PRETRAINED = not CFG["smoke_test"]
S = CFG["image_size"]
snr = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"], ppd=CFG["ppd"])
grid = FixationGrid(2, 2, CFG["margin"])
fixations = grid.get_fixations()

MU  = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class FullPipeline(nn.Module):
    """Full foveated model: fov_transform -> backbone -> (optional multi-glance).

    Accepts raw [0,1] pixel tensors. Applies foveation then backbone normalisation.
    """
    def __init__(self, fov_transform, backbone, norm, fixations_list=None,
                 halt_threshold=0.75, sigma0=0.5, slope=1.5, ppd=4.0,
                 multi_glance=False):
        super().__init__()
        self.fov = fov_transform     # None -> no foveation
        self.backbone = models.NormalizedModel(backbone, norm)
        self.multi_glance = multi_glance
        self.fixations = fixations_list or [(0.5, 0.5)]
        self.halt_threshold = halt_threshold
        self.sigma0 = sigma0; self.slope = slope; self.ppd = ppd

    def apply_fov(self, x_raw, fixation_yx=None):
        if self.fov is None:
            return x_raw
        # Temporarily override fixation in fov transform's periphery modules
        if fixation_yx is not None and hasattr(self.fov, 'fixation_yx'):
            self.fov.fixation_yx = fixation_yx
            if self.fov.periphery is not None:
                self.fov.periphery.fixation_yx = fixation_yx
        return torch.stack([self.fov(x_raw[i]) for i in range(x_raw.size(0))])

    def forward(self, x_raw, return_glance_count=False):
        B = x_raw.size(0)
        if not self.multi_glance:
            fov_x = self.apply_fov(x_raw)
            logits = self.backbone(fov_x)
            if return_glance_count:
                return logits, torch.ones(B, dtype=torch.long, device=x_raw.device)
            return logits

        logit_sum = None
        halted = torch.zeros(B, dtype=torch.bool, device=x_raw.device)
        gc = torch.ones(B, dtype=torch.long, device=x_raw.device)
        for t, fix in enumerate(self.fixations):
            fov_x = self.apply_fov(x_raw, fix)
            z = self.backbone(fov_x)
            logit_sum = z if logit_sum is None else logit_sum + z
            if not self.training and t < len(self.fixations) - 1:
                conf = F.softmax(logit_sum / (t+1), dim=1).max(1).values
                new_halt = (conf >= self.halt_threshold) & (~halted)
                gc += (~halted & ~new_halt).long()
                halted |= new_halt
                if halted.all(): break
        final = logit_sum / len(self.fixations)
        return (final, gc) if return_glance_count else final


def build_variant(label, PRETRAINED, snr, fixations, CFG):
    """Build one ablation variant. Returns FullPipeline."""
    # Default: full model
    use_fov    = True   # C1
    use_trace  = True   # C2
    use_vnoise = True   # C3
    use_cornet = False  # C4 variant (True = CORnet-S instead of ResNet-50)
    use_multi  = True   # C5

    if   label == "full":         pass
    elif label == "no_C1_fov":    use_fov    = False
    elif label == "no_C2_trace":  use_trace  = False
    elif label == "no_C3_noise":  use_vnoise = False
    elif label == "no_C4_resnet": use_cornet = True
    elif label == "no_C5_multi":  use_multi  = False

    # Back-end
    if use_cornet:
        backbone, norm = models.build_cornet_s(pretrained=PRETRAINED)
    else:
        backbone, norm = models.build_voneresnet50(pretrained=PRETRAINED)
        set_v1_noise_mode(backbone, mode="neuronal" if use_vnoise else None)

    backbone.eval()

    # Foveation transform
    if not use_fov:
        ft = None
    elif not use_trace:
        ft = build_foveated_transform("blur", snr, CFG["patch_size"],
                                       CFG["fovea_deg"], CFG["transition_deg"],
                                       CFG["ppd"], (0.5, 0.5),
                                       CFG["rblur_sigma0"], CFG["rblur_slope"])
    else:
        ft = build_foveated_transform("trace", snr, CFG["patch_size"],
                                       CFG["fovea_deg"], CFG["transition_deg"],
                                       CFG["ppd"], (0.5, 0.5))

    return FullPipeline(ft, backbone, norm,
                        fixations_list=fixations if use_multi else [(0.5, 0.5)],
                        halt_threshold=CFG["halt_threshold"],
                        sigma0=CFG["rblur_sigma0"],
                        slope=CFG["rblur_slope"],
                        ppd=CFG["ppd"],
                        multi_glance=use_multi)


ABLATION_LABELS = ["full", "no_C1_fov", "no_C2_trace",
                    "no_C3_noise", "no_C4_resnet", "no_C5_multi"]

print("Building ablation variants...")
ablation_models = {}
for lbl in ABLATION_LABELS:
    ablation_models[lbl] = build_variant(lbl, PRETRAINED, snr, fixations, CFG).to(device)
    n_params = sum(p.numel() for p in ablation_models[lbl].parameters()) / 1e6
    print(f"  {lbl:20s}  {n_params:.1f}M params")

In [ ]:
# ── Real CIFAR-10 + training / evaluation ─────────────────────────────────
# Honest fallback: src/datasets.py::get_cifar10 returns a clearly-labelled synthetic
# stand-in only if smoke_test=True and CIFAR-10 hasn't been downloaded yet.

train_ds = datasets.get_cifar10(CFG["data_dir"], train=True, normalization="imagenet",
                                 download=True, smoke_test=CFG["smoke_test"])
val_ds   = datasets.get_cifar10(CFG["data_dir"], train=False, normalization="imagenet",
                                 download=True, smoke_test=CFG["smoke_test"])
_num_workers = 2 if not CFG["smoke_test"] else 0
train_ld = torch.utils.data.DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                                        num_workers=_num_workers)
val_ld   = torch.utils.data.DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                                        num_workers=_num_workers)


class AblationTrainer(nn.Module):
    """Frozen FullPipeline backbone + a trainable linear head mapping its own output
    dimension (e.g. 1000 for an ImageNet-pretrained VOneResNet-50/CORnet-S backbone)
    down to CFG['n_classes'] (CIFAR-10) -- a fast linear probe, matching
    03_v1_block.ipynb's methodology, rather than full end-to-end fine-tuning (that is
    what 07_cifar10_and_cifar10c_evaluation.ipynb already does for the 5 core
    conditions; this notebook's job is the full leave-one-out comparison, cheaply).
    """
    def __init__(self, pipeline, n_classes, image_size=32):
        super().__init__()
        self.pipeline = pipeline
        # Detect output dimension from a dummy forward pass
        with torch.no_grad():
            _dummy_out = pipeline(torch.rand(1, 3, image_size, image_size).to(
                next(pipeline.parameters()).device))
        out_dim = _dummy_out.shape[-1]   # e.g. 1000 for ImageNet backbone
        self.head = nn.Linear(out_dim, n_classes)

    def forward(self, x_raw, return_glance_count=False):
        out = self.pipeline(x_raw, return_glance_count=return_glance_count)
        if return_glance_count:
            logits, gc = out
            return self.head(logits), gc
        return self.head(out)


def train_ablation(pipeline, n_classes, loader, n_batches, n_epochs=1):
    """Train only `trainer.head` (pipeline stays frozen) for `n_epochs`, capping
    each epoch at `n_batches` batches (`None` = full loader)."""
    trainer = AblationTrainer(pipeline, n_classes, CFG["image_size"]).to(device)
    opt    = torch.optim.Adam(trainer.head.parameters(), lr=CFG["lr"])
    trainer.train()
    for _epoch in range(n_epochs):
        for bi, (xb, yb) in enumerate(loader):
            if n_batches is not None and bi >= n_batches: break
            xb, yb = xb.to(device), yb.to(device)
            x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
            logits = trainer(x_raw)
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return trainer


@torch.no_grad()
def eval_pipeline(trainer, loader, n_batches=None):
    """Evaluate a trained `AblationTrainer` (head-adapted, CFG['n_classes']-way)."""
    trainer.eval()
    correct = total = total_gc = 0
    for bi, (xb, yb) in enumerate(loader):
        if n_batches is not None and bi >= n_batches: break
        xb, yb = xb.to(device), yb.to(device)
        x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
        logits, gc = trainer(x_raw, return_glance_count=True)
        total_gc += gc.sum().item()
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    return correct / max(total, 1), total_gc / max(total, 1)

In [ ]:
# ── CIFAR-10-C: detailed corruption breakdown (shared by leave-one-out + beta sweep) ──
# Per-corruption-type AND per-category (noise/blur/weather/digital, Hendrycks &
# Dietterich 2019) error rates, not just a single averaged number.

CIFAR10C_CORRUPTIONS = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "glass_blur", "motion_blur", "zoom_blur",
    "snow", "frost", "fog", "brightness",
    "contrast", "elastic_transform", "pixelate", "jpeg_compression",
]
CIFAR10C_CATEGORIES = {
    "noise":   ["gaussian_noise", "shot_noise", "impulse_noise"],
    "blur":    ["defocus_blur", "glass_blur", "motion_blur", "zoom_blur"],
    "weather": ["snow", "frost", "fog", "brightness"],
    "digital": ["contrast", "elastic_transform", "pixelate", "jpeg_compression"],
}

os.makedirs(CFG["cifar10c_dir"], exist_ok=True)
if not CFG["smoke_test"]:
    _c10c_check = os.path.join(CFG["cifar10c_dir"], "gaussian_noise.npy")
    if not os.path.exists(_c10c_check):
        print("Downloading CIFAR-10-C from Zenodo (~270 MB)...")
        import urllib.request, tarfile
        _tmp = "/tmp/CIFAR-10-C.tar"
        urllib.request.urlretrieve("https://zenodo.org/record/2535967/files/CIFAR-10-C.tar", _tmp)
        with tarfile.open(_tmp) as tf:
            tf.extractall(CFG["data_dir"])
        _extracted = os.path.join(CFG["data_dir"], "CIFAR-10-C")
        if os.path.isdir(_extracted):
            os.rename(_extracted, CFG["cifar10c_dir"])
        print("CIFAR-10-C download complete.")
    else:
        print("CIFAR-10-C already on disk.")
else:
    print("smoke_test=True: skipping CIFAR-10-C download.")


class CIFAR10C(torch.utils.data.Dataset):
    """CIFAR-10-C corrupted test images (Hendrycks et al. 2019), returned already
    ImageNet-normalised (matches what train_ld/val_ld/eval_pipeline expect)."""
    def __init__(self, root, corruption, severity):
        c_path = os.path.join(root, f"{corruption}.npy")
        l_path = os.path.join(root, "labels.npy")
        if not os.path.exists(c_path):
            raise FileNotFoundError(f"CIFAR-10-C not found: {c_path}")
        offset = (severity - 1) * 10000
        self.images = np.load(c_path)[offset: offset + 10000]   # [N,32,32,3] uint8
        self.labels = np.load(l_path)[offset: offset + 10000].astype(int)

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i].astype(np.float32) / 255.0).permute(2, 0, 1)
        img = (img - MU.view(3, 1, 1)) / STD.view(3, 1, 1)
        return img, int(self.labels[i])


def eval_corruption_breakdown(trainer, root, n_batches=None, severities=(1, 3, 5)):
    """Per-corruption-type and per-category top-1 error rates.

    Returns {"per_corruption": {15 entries}, "per_category": {4 entries},
             "overall": scalar}. NaN throughout if CIFAR-10-C isn't present
    (smoke_test / not yet downloaded) -- honest fallback, no crash.

    NOTE: this is NOT AlexNet-normalised like notebook 07's mCE (06 doesn't build
    an AlexNet reference) -- a plain error rate is directly comparable across the
    conditions/betas compared here, which is all these two experiments need.
    """
    per_corruption = {}
    for corruption in CIFAR10C_CORRUPTIONS:
        errs = []
        for severity in severities:
            try:
                ds = CIFAR10C(root, corruption, severity)
            except FileNotFoundError:
                errs = None
                break
            ld = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False)
            acc, _ = eval_pipeline(trainer, ld, n_batches=n_batches)
            errs.append(1.0 - acc)
        per_corruption[corruption] = float(np.mean(errs)) if errs else float("nan")

    per_category = {}
    for cat, corruptions in CIFAR10C_CATEGORIES.items():
        vals = [per_corruption[c] for c in corruptions]
        per_category[cat] = float(np.nanmean(vals)) if not all(np.isnan(v) for v in vals) else float("nan")

    overall_vals = list(per_corruption.values())
    overall = (float(np.nanmean(overall_vals))
               if not all(np.isnan(v) for v in overall_vals) else float("nan"))

    return {"per_corruption": per_corruption, "per_category": per_category, "overall": overall}


def print_corruption_breakdown(label, breakdown):
    """Detailed console report: per-category, then per-corruption-type, then overall."""
    print(f"\n  {label} -- CIFAR-10-C breakdown")
    print(f"    {'Category':<10} {'Error':>8}")
    for cat, err in breakdown["per_category"].items():
        print(f"    {cat:<10} {err:>8.4f}")
    print(f"    {'-'*19}")
    print(f"    {'Overall':<10} {breakdown['overall']:>8.4f}")
    print(f"    Per-corruption-type:")
    for cat, corruptions in CIFAR10C_CATEGORIES.items():
        for c in corruptions:
            print(f"      [{cat[:4]}] {c:<20} {breakdown['per_corruption'][c]:>8.4f}")

In [ ]:
# ── Run leave-one-out ablation ────────────────────────────────────────────

n_epochs      = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches     = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]
n_batches_val = 10 if CFG["smoke_test"] else None
n_batches_c10c      = CFG["n_batches_c10c_smoke"] if CFG["smoke_test"] else None
c10c_severities_loo = (3,) if CFG["smoke_test"] else (1, 3, 5)

print("Running leave-one-out ablation sweep"
      + (" (smoke_test)..." if CFG["smoke_test"] else " (real CIFAR-10)..."))
print(f"{'Variant':20s}  {'Clean acc':>10}  {'Avg glances':>12}  {'Efficiency':>10}")
print("-" * 58)

ablation_results = {}
trainers = {}
for lbl, pipeline in ablation_models.items():
    common.set_seed(CFG["seed"])
    # Train the linear head (pipeline stays frozen) on real CIFAR-10
    trainer = train_ablation(pipeline, CFG["n_classes"], train_ld, n_batches, n_epochs=n_epochs)
    trainers[lbl] = trainer

    val_acc, avg_gc = eval_pipeline(trainer, val_ld, n_batches=n_batches_val)
    efficiency = val_acc / max(avg_gc, 1)
    c10c_breakdown = eval_corruption_breakdown(trainer, CFG["cifar10c_dir"],
                                                n_batches=n_batches_c10c,
                                                severities=c10c_severities_loo)

    ablation_results[lbl] = {
        "val_acc": val_acc,
        "avg_glances": avg_gc,
        "efficiency": efficiency,
        "c10c_breakdown": c10c_breakdown,
    }
    print(f"  {lbl:20s}  {val_acc:10.4f}  {avg_gc:12.2f}  {efficiency:10.4f}")
    print_corruption_breakdown(lbl, c10c_breakdown)

# Compute marginal contributions relative to full model
full_acc = ablation_results["full"]["val_acc"]
print(f"\n{'Marginal contribution Delta_k = full - ablated':}")
for lbl, res in ablation_results.items():
    if lbl == "full": continue
    delta = full_acc - res["val_acc"]
    print(f"  {lbl:20s}  Delta_k = {delta:+.4f}")

if CFG["smoke_test"]:
    print("\n(smoke_test=True: real CIFAR-10 if present, else a clearly-labelled "
          "synthetic pipeline check -- see datasets.py honest-fallback policy; few "
          "epochs/batches means deltas are noisy. Set smoke_test=False for real results. "
          "CIFAR-10-C breakdown is NaN until smoke_test=False downloads it.)")


# ── Figure: per-category CIFAR-10-C error, grouped by leave-one-out condition ──
_cond_labels = list(ablation_results.keys())
_categories  = list(CIFAR10C_CATEGORIES.keys())
_x = np.arange(len(_cond_labels))
_width = 0.8 / len(_categories)

fig_c10c, ax_c10c = plt.subplots(figsize=(10, 5))
for ci, cat in enumerate(_categories):
    vals = [ablation_results[lbl]["c10c_breakdown"]["per_category"][cat] for lbl in _cond_labels]
    ax_c10c.bar(_x + ci * _width, vals, _width, label=cat)
ax_c10c.set_xticks(_x + _width * (len(_categories) - 1) / 2)
ax_c10c.set_xticklabels([l.replace("_", " ") for l in _cond_labels], rotation=20, ha="right")
ax_c10c.set_ylabel("Top-1 error rate")
ax_c10c.set_title("CIFAR-10-C error by category, per leave-one-out condition\n"
                   + ("(smoke_test: NaN until real CIFAR-10-C is downloaded)"
                      if CFG["smoke_test"] else "(real CIFAR-10-C)"))
ax_c10c.legend(title="corruption category", fontsize=8)
ax_c10c.grid(axis="y", alpha=0.3)
plt.tight_layout()
c10c_cat_path = os.path.join(CFG["result_dir"], "06_ablation_c10c_by_category.png")
common.save_figure(fig_c10c, c10c_cat_path, dpi=130)
plt.close(fig_c10c)
print(f"\nSaved: {c10c_cat_path}")

In [ ]:
# ── Adversarial robustness for full model ─────────────────────────────────

print("PGD robustness evaluation (full model, EOT)...")

# NOTE: the raw `ablation_models["full"]` pipeline outputs its backbone's native
# class count (e.g. 1000 for ImageNet-pretrained VOneResNet-50), not CIFAR-10's 10 --
# comparing its argmax against CIFAR-10 labels directly would be a category mismatch.
# `trainers["full"]` (built in the leave-one-out cell above) is the same frozen
# pipeline plus the trained CIFAR-10 linear head, so use that instead.
full_trainer = trainers["full"]

# Small batch from val set
xb, yb = next(iter(val_ld))
xb, yb = xb.to(device), yb.to(device)
x_raw  = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)

# PGD against the full trainer (pipeline + CIFAR-10 head)
# EOT-N: needed because C3 (VOneBlock Poisson) makes the forward stochastic
x_adv = pgd_attack(full_trainer, x_raw, yb,
                    eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                    steps=CFG["pgd_steps"], eot_samples=CFG["eot_samples"])

with torch.no_grad():
    clean_logits = full_trainer(x_raw)
    adv_logits   = full_trainer(x_adv)

clean_acc = (clean_logits.argmax(1) == yb).float().mean().item()
adv_acc   = (adv_logits.argmax(1)   == yb).float().mean().item()
print(f"  Clean accuracy  : {clean_acc:.4f}")
print(f"  Robust accuracy : {adv_acc:.4f}  (PGD eps={CFG['eps']:.4f}, EOT={CFG['eot_samples']})")
print("  (smoke_test=True: few epochs on possibly-synthetic data -> not meaningful yet.)"
      if CFG["smoke_test"] else "")

In [ ]:
# ── Ablation table figure ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels_short = [l.replace("no_", "-").replace("_", " ") for l in ABLATION_LABELS]
accs = [ablation_results[l]["val_acc"]     for l in ABLATION_LABELS]
gcs  = [ablation_results[l]["avg_glances"] for l in ABLATION_LABELS]
effs = [ablation_results[l]["efficiency"]  for l in ABLATION_LABELS]

colors = plt.cm.Set2(np.linspace(0, 1, len(ABLATION_LABELS)))

ax = axes[0]
bars = ax.barh(range(len(ABLATION_LABELS)), accs, color=colors)
ax.set_yticks(range(len(ABLATION_LABELS)))
ax.set_yticklabels(labels_short, fontsize=9)
ax.set_xlabel("Val accuracy")
ax.set_title("Leave-one-out ablation: clean accuracy\n(smoke_test — not meaningful)")
ax.axvline(accs[0], color="k", lw=1, ls="--", alpha=0.5, label="full model")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=0.3)

ax = axes[1]
ax.scatter(gcs, accs, c=range(len(ABLATION_LABELS)), cmap="Set2", s=120, zorder=5)
for i, lbl in enumerate(labels_short):
    ax.annotate(lbl, (gcs[i], accs[i]), textcoords="offset points",
                xytext=(5, 3), fontsize=7)
ax.set_xlabel("Avg glances per image")
ax.set_ylabel("Val accuracy")
ax.set_title("Efficiency curve: accuracy vs glances\n(each point = one ablation condition)")
ax.grid(True, alpha=0.3)

fig.suptitle(
    "Full model leave-one-out ablation matrix\n"
    "(smoke_test=True: random weights + synthetic data — for pipeline verification only)",
    fontsize=9, y=1.02,
)
plt.tight_layout()
tbl_path = os.path.join(CFG["result_dir"], "06_ablation_table.png")
common.save_figure(fig, tbl_path, dpi=130)
plt.close(fig)
print(f"Saved: {tbl_path}")


In [ ]:
# ── Beta sweep for the "full" model (VOneNet + trace periphery + multi-glance) ──
# Tests whether the SNR/beta "sweet spot" (docs Ablation A6) still holds once
# VOneNet noise (C3) and IT-feedback multi-glance (C5) are combined with the
# trace-based periphery (C2). Notebook 07 already sweeps beta for a *plain*
# ResNet-50 + trace periphery (no VOneNet/multi-glance) -- this repeats that
# sweep for the full combined pipeline. beta=1-3 alone barely change alpha(r)
# within CIFAR-10's 8deg field of view (see 02_foveation_rblur_and_periphery.ipynb's
# beta-sweep figure), so the range is widened to include 5, 8.
# CIFAR10C_CORRUPTIONS/CATEGORIES, the CIFAR10C dataset, and
# eval_corruption_breakdown/print_corruption_breakdown are defined earlier
# (shared with the leave-one-out sweep above).

n_epochs_beta       = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches_beta       = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]
n_batches_val_beta   = 10 if CFG["smoke_test"] else None
n_batches_c10c_beta  = CFG["n_batches_c10c_smoke"] if CFG["smoke_test"] else None
c10c_severities_beta = (3,) if CFG["smoke_test"] else (1, 3, 5)

print("Running beta sweep for the full model (VOneNet + trace periphery + multi-glance)...")
print(f"{'beta':>6}  {'val_acc':>10}  {'avg_c10c_err':>13}  {'robust_acc':>11}")
print("-" * 48)

beta_sweep_results = {}
for beta_val in CFG["beta_sweep"]:
    common.set_seed(CFG["seed"])
    snr_beta = SNRProfile(snr0_db=CFG["snr0_db"], beta=beta_val, ppd=CFG["ppd"])
    full_pipeline_beta = build_variant("full", PRETRAINED, snr_beta, fixations, CFG).to(device)

    trainer_beta = train_ablation(full_pipeline_beta, CFG["n_classes"], train_ld,
                                   n_batches_beta, n_epochs=n_epochs_beta)
    val_acc, avg_gc = eval_pipeline(trainer_beta, val_ld, n_batches=n_batches_val_beta)
    c10c_breakdown = eval_corruption_breakdown(trainer_beta, CFG["cifar10c_dir"],
                                                n_batches=n_batches_c10c_beta,
                                                severities=c10c_severities_beta)

    xb, yb = next(iter(val_ld))
    xb, yb = xb.to(device), yb.to(device)
    x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
    x_adv = pgd_attack(trainer_beta, x_raw, yb,
                        eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                        steps=CFG["pgd_steps"], eot_samples=CFG["eot_samples"])
    with torch.no_grad():
        robust_acc = (trainer_beta(x_adv).argmax(1) == yb).float().mean().item()

    beta_sweep_results[f"beta{beta_val:.0f}"] = {
        "beta": beta_val, "val_acc": val_acc,
        "c10c_breakdown": c10c_breakdown, "robust_acc": robust_acc,
    }
    print(f"{beta_val:6.1f}  {val_acc:10.4f}  {c10c_breakdown['overall']:13.4f}  {robust_acc:11.4f}")
    print_corruption_breakdown(f"beta={beta_val:g}", c10c_breakdown)

print("\n(c10c error: NOT AlexNet-normalised like 07's mCE, so compare only within "
      "this sweep, not directly against notebook 07's numbers.)")
if CFG["smoke_test"]:
    print("(smoke_test=True: few epochs/batches -> not meaningful yet; "
          "CIFAR-10-C not downloaded until smoke_test=False.)")

# Figure: beta vs clean accuracy / per-category + overall C10C error / PGD robust acc
fig3, ax3s = plt.subplots(1, 3, figsize=(16, 4.5))
betas_sorted = sorted(beta_sweep_results, key=lambda k: beta_sweep_results[k]["beta"])
xs = [beta_sweep_results[k]["beta"] for k in betas_sorted]

ax = ax3s[0]
ax.plot(xs, [beta_sweep_results[k]["val_acc"] for k in betas_sorted], "o-", color="steelblue", markersize=8)
ax.set_xlabel("beta"); ax.set_ylabel("val_acc")
ax.set_title("Clean CIFAR-10 accuracy\n(higher = better)", fontsize=9)
ax.grid(True, alpha=0.3)

ax = ax3s[1]
for cat in CIFAR10C_CATEGORIES:
    ys = [beta_sweep_results[k]["c10c_breakdown"]["per_category"][cat] for k in betas_sorted]
    ax.plot(xs, ys, "o--", markersize=5, alpha=0.7, label=cat)
ax.plot(xs, [beta_sweep_results[k]["c10c_breakdown"]["overall"] for k in betas_sorted],
        "o-", color="black", markersize=8, linewidth=2, label="overall")
ax.set_xlabel("beta"); ax.set_ylabel("error rate")
ax.set_title("CIFAR-10-C error, by category\n(lower = better)", fontsize=9)
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

ax = ax3s[2]
ax.plot(xs, [beta_sweep_results[k]["robust_acc"] for k in betas_sorted], "o-", color="steelblue", markersize=8)
ax.set_xlabel("beta"); ax.set_ylabel("robust_acc")
ax.set_title(f"PGD robust accuracy (eps={CFG['eps']:.3f})\n(higher = better)", fontsize=9)
ax.grid(True, alpha=0.3)

fig3.suptitle("Full model (VOneNet + trace periphery + multi-glance): beta sweep",
              fontsize=11, y=1.03)
plt.tight_layout()
beta_fig_path = os.path.join(CFG["result_dir"], "06_full_model_beta_sweep.png")
common.save_figure(fig3, beta_fig_path, dpi=130)
plt.close(fig3)
print(f"Saved: {beta_fig_path}")

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────
summary = {
    "notebook": "06_full_model_and_ablations",
    "cfg":      {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "ablation_table": ablation_results,
    "adversarial": {
        "clean_acc": clean_acc, "robust_acc": adv_acc,
        "eps": CFG["eps"], "eot_samples": CFG["eot_samples"],
    },
    "full_model_beta_sweep": beta_sweep_results,
    "note": "smoke_test=True: all numbers reflect pipeline, not real performance.",
}
rpath = os.path.join(CFG["result_dir"], "06_ablation_table.json")
common.save_json(summary, rpath)
common.save_config(CFG, os.path.join(CFG["result_dir"], "06_config.json"))

print(f"Results: {rpath}")
print(f"06_ablation_table.png : {os.path.exists(tbl_path)}")
print(f"06_full_model_beta_sweep.png : {os.path.exists(beta_fig_path)}")
print()
print("=" * 60)
print("ALL NOTEBOOKS COMPLETE")
print("=" * 60)
nbs = ["00_setup_and_data", "01_baseline_reproduce",
       "02_foveation_rblur_and_periphery", "03_v1_block",
       "04_it_feedback_multiglance", "05_mftma_certification",
       "06_full_model_and_ablations"]
for nb in nbs:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "notebooks", nb + ".ipynb"))
    print(f"  {'OK' if exists else 'MISSING':6s} notebooks/{nb}.ipynb")
print()
src_files = ["common.py", "datasets.py", "eval_harness.py", "models.py",
             "overrides.py", "foveation.py", "it_feedback.py", "mftma.py"]
for sf in src_files:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "src", sf))
    print(f"  {'OK' if exists else 'MISSING':6s} src/{sf}")
